# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Two findings are worth scrutinizing here.

1. The model is intended to rank pages that look like good refresh candidates. The label is derived from the traffic trend, specifically whether the page moved down relative to its recent history, so the claim is a prediction/ranking claim rather than a causal claim about refresh impact.
2. Visibility and freshness-related signals are useful for this ranking task. That is a reasonable finding, but it should be read as "these features were associated with the label in this dataset" rather than "these factors cause decline." The validation design carries the ranking claim when it is evaluated on held-out data, but it does not prove intervention effects.

The core methodology question is whether the evaluation is honest enough for the claim. A client-aware split is more appropriate than a random row split for a problem where multiple rows from the same client can share hidden context.


In [ ]:
import json
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts' / 'ml_utils.py').exists() and (candidate / 'data' / 'processed' / 'refresh_feature_vector.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current notebook path.')


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

frame = pd.read_csv(ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv')
frame['is_declining_label'] = frame['trend_direction'].str.lower().eq('down').astype(int)

numeric_frame = frame[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)
categorical_frame = frame[MODEL_CATEGORICAL_FEATURES].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=MODEL_CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
feature_frame = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)

X = feature_frame
y = frame['is_declining_label']
clients = frame['client_id'].fillna('unknown').astype(str)


def metric_table(y_true, scores):
    preds = (np.asarray(scores) >= 0.5).astype(int)
    return {
        'base_rate': float(y_true.mean()),
        'accuracy': accuracy_score(y_true, preds),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1': f1_score(y_true, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_true, np.asarray(scores)),
        'average_precision': average_precision_score(y_true, np.asarray(scores)),
        'precision_at_20': precision_at_k(y_true, scores, 20),
        'precision_at_50': precision_at_k(y_true, scores, 50),
        'precision_at_100': precision_at_k(y_true, scores, 100),
    }


random_idx = np.arange(len(frame))
train_idx, test_idx = train_test_split(random_idx, test_size=0.2, random_state=42, stratify=y)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_g, test_g = next(gss.split(X, y, groups=clients))

model = RandomForestClassifier(
    class_weight='balanced_subsample',
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)
model.fit(X.iloc[train_idx], y.iloc[train_idx])
random_probs = model.predict_proba(X.iloc[test_idx])[:, 1]
random_metrics = metric_table(y.iloc[test_idx], random_probs)

model_grouped = RandomForestClassifier(
    class_weight='balanced_subsample',
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)
model_grouped.fit(X.iloc[train_g], y.iloc[train_g])
grouped_probs = model_grouped.predict_proba(X.iloc[test_g])[:, 1]
grouped_metrics = metric_table(y.iloc[test_g], grouped_probs)

leak_frame = X.copy()
leak_frame['is_declining_label_leak'] = y.to_numpy()
model_leak = RandomForestClassifier(
    class_weight='balanced_subsample',
    max_depth=10,
    min_samples_leaf=25,
    n_estimators=200,
    n_jobs=-1,
    random_state=42,
)
model_leak.fit(leak_frame.iloc[train_g], y.iloc[train_g])
leak_probs = model_leak.predict_proba(leak_frame.iloc[test_g])[:, 1]
leak_metrics = metric_table(y.iloc[test_g], leak_probs)

summary = {
    'random_split': random_metrics,
    'grouped_split': grouped_metrics,
    'leakage_stress_test': leak_metrics,
}
print(json.dumps(summary, indent=2))


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

The random-split result is strong, but the grouped split is much weaker. That gap is the main finding here: the model is learning client-specific patterns that do not generalize as cleanly to unseen clients.

The verified numbers are:

- Random split: precision@50 = 0.90, ROC AUC = 0.758, average precision = 0.769
- Grouped split: precision@50 = 0.56, ROC AUC = 0.609, average precision = 0.589
- Base rate in the grouped test set: 0.511

These results show that the earlier performance was optimistic under a less honest split.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the final feature set against the three leakage patterns from the skill guide.

- Label-derived features: the feature matrix excludes `trend_direction`, `trend_pct`, `impressions_last_30d`, `clicks_last_30d`, and `sessions_last_30d`. Those are direct label sources or part of the label formula.
- Future/overlapping windows: the feature set uses historical 90-day aggregates and does not include the final 30-day windows that define the label.
- Decision-derived features: I did not include product flags or existing-system score columns in the model features.

The leakage stress test is especially telling. When I add a single obvious label leak column, the model becomes nearly perfect on the grouped test set: precision@50 rises to 1.00 and ROC AUC to 1.00. That confirms the test harness is sensitive to leakage and that the honest grouped result is not an artifact.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Original claim: "This model will reliably identify the best pages to refresh."

Safer rewrite: "In this dataset, the model was observed to rank pages with declining traffic more effectively than a simple baseline, and the result is best used as a directional decision-support signal for human review rather than as a fully reliable automation tool."


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.